# Capítulo 13 · Amostragem e estimação: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, a lista de escolares de um município e um levantamento de saúde bucal, pelas dez perguntas do secretário. A **Parte B** traz o código e o resultado de cada exercício do capítulo (os enunciados estão no livro). Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/schoolchildren.csv`: 14.299 linhas, uma por criança matriculada, e quatro colunas: `child_id`, `school` (S01 a S48), `school_type` (`private` ou `public`), que a lista da secretaria de educação traz, e `dmft`, que na vida real só é conhecido depois do exame e aqui vem para todas as crianças, para que cada método possa ser conferido contra a verdade. Cada célula traz o número da seção do livro a que pertence.

### 13.1 Introdução: os dados

In [1]:
# 1. Ler o arquivo: cada linha é uma criança matriculada (o dmft só é conhecido, na vida real, depois do exame)
import numpy as np
import pandas as pd
from math import sqrt
from scipy import stats

df = pd.read_csv("data/schoolchildren.csv")

# 2. Contar linhas e colunas
rows, columns = df.shape
print("A lista tem", rows, "linhas (uma por criança) e", columns, "colunas.")
df.head()                                      # As cinco primeiras linhas

A lista tem 14299 linhas (uma por criança) e 4 colunas.


,child_id,school,school_type,dmft
0,K00001,S01,private,5
1,K00002,S01,private,2
2,K00003,S01,private,3
3,K00004,S01,private,0
4,K00005,S01,private,2


### 13.2 Quem o levantamento descreve?

*Quem o levantamento descreve, e o modo de escolher as crianças muda o significado do resultado?*

In [2]:
# 1. A lista inteira: média populacional, desvio padrão e parcela livre de cárie
dmft = df["dmft"].values
N = len(dmft)
mu = dmft.mean()
sigma = dmft.std(ddof=0)
print("População: média", round(mu, 3), "; desvio padrão", round(sigma, 3), "; livres de cárie:", round(100 * (dmft == 0).mean(), 1), "%;", N, "crianças")

# 2. Por tipo de escola
display(df.groupby("school_type")["dmft"].agg(["size", "mean", "std"]).round(3))

# 3. Uma amostra de conveniência: as primeiras 600 crianças da lista
first600 = df.iloc[:600]
print("Amostra de conveniência: média", round(first600["dmft"].mean(), 3), "; livres de cárie:", round(100 * (first600["dmft"] == 0).mean(), 1), "%; escolas:", first600["school"].nunique())

População: média 1.87 ; desvio padrão 2.389 ; livres de cárie: 35.7 %; 14299 crianças


,size,mean,std
school_type,,,
private,3448,0.997,1.416
public,10851,2.148,2.562


Amostra de conveniência: média 1.295 ; livres de cárie: 44.0 %; escolas: 3


### 13.3 Sorteando as crianças

*Como sortear 600 crianças ao acaso, e com que outras regras se pode tirar uma amostra probabilística?*

In [3]:
# 1. Uma amostra aleatória simples de 600 crianças (semente 13)
srs = df.sample(n=600, random_state=13)
print("Amostra aleatória simples: média", round(srs["dmft"].mean(), 3), "; escolas:", srs["school"].nunique(), "; particulares:", round(100 * (srs["school_type"] == "private").mean(), 1), "%")

# 2. Amostra sistemática: uma criança a cada 24 a partir de um início sorteado, repetida 5.000 vezes
rng = np.random.default_rng(13)
systematic_means = []
for repetition in range(5000):
    start = rng.integers(0, 24)
    systematic_means.append(dmft[np.arange(start, N, 24)].mean())
srs_se = sigma / sqrt(600) * sqrt((N - 600) / (N - 1))
print("Sistemática: desvio padrão das médias", round(np.std(systematic_means), 4), "; erro padrão da amostra aleatória simples", round(srs_se, 4))

Amostra aleatória simples: média 1.847 ; escolas: 48 ; particulares: 24.3 %
Sistemática: desvio padrão das médias 0.0841 ; erro padrão da amostra aleatória simples 0.0954


### 13.4 Quanto a média muda de amostra para amostra?

*Quanto o dmft médio mudaria de uma amostra para outra?*

In [4]:
# 1. 2.000 amostras aleatórias simples de 600 crianças: a distribuição amostral da média
rng = np.random.default_rng(13)
means600 = []
for repetition in range(2000):
    means600.append(dmft[rng.choice(N, 600, replace=False)].mean())
means600 = np.array(means600)
print("Médias: média", round(means600.mean(), 3), "; desvio padrão das médias", round(means600.std(), 4))

# 2. Erro padrão pela fórmula, sem e com a correção para população finita
fpc = sqrt((N - 600) / (N - 1))
print("Erro padrão: sigma/raiz(n) =", round(sigma / sqrt(600), 4), "; com a correção", round(sigma / sqrt(600) * fpc, 4), "; fator de correção", round(fpc, 3))

# 3. Variância dividindo por n e por n - 1, em 5.000 amostras de 10 crianças
rng = np.random.default_rng(13)
variances_n = []
variances_n1 = []
for repetition in range(5000):
    small = dmft[rng.choice(N, 10, replace=False)]
    variances_n.append(small.var(ddof=0))
    variances_n1.append(small.var(ddof=1))
print("Variância populacional", round(sigma ** 2, 3), "; dividindo por n", round(np.mean(variances_n), 3), "; dividindo por n - 1", round(np.mean(variances_n1), 3))

# 4. Média das n primeiras crianças em 20 ordens aleatórias da lista
rng = np.random.default_rng(13)
paths = []
for order in range(20):
    paths.append(np.cumsum(dmft[rng.permutation(N)]) / np.arange(1, N + 1))
paths = np.array(paths)
for n_children in (30, 600, 3000):
    column = paths[:, n_children - 1]
    print("Número de crianças:", n_children, "; as 20 médias vão de", round(column.min(), 2), "até", round(column.max(), 2))

# 5. Meia-largura da média populacional mais e menos 1,96 erros padrão
for n_children in (600, 3000):
    half_width = 1.96 * sigma / sqrt(n_children) * sqrt((N - n_children) / (N - 1))
    print("Número de crianças:", n_children, "; meia-largura da faixa (1,96 erros padrão):", round(half_width, 2))

Médias: média 1.873 ; desvio padrão das médias 0.093
Erro padrão: sigma/raiz(n) = 0.0975 ; com a correção 0.0954 ; fator de correção 0.979
Variância populacional 5.705 ; dividindo por n 5.133 ; dividindo por n - 1 5.704
Número de crianças: 30 ; as 20 médias vão de 1.3 até 2.63
Número de crianças: 600 ; as 20 médias vão de 1.72 até 2.04
Número de crianças: 3000 ; as 20 médias vão de 1.82 até 1.95
Número de crianças: 600 ; meia-largura da faixa (1,96 erros padrão): 0.19
Número de crianças: 3000 ; meia-largura da faixa (1,96 erros padrão): 0.08


### 13.5 Qual é o formato da variação da média?

*As médias das amostras se espalham em forma de sino, mesmo que o dmft das crianças não tenha esse formato?*

In [5]:
# 1. Médias amostrais para amostras de 5, 30 e 600 crianças (2.000 amostras cada)
for n_children in (5, 30, 600):
    rng = np.random.default_rng(13)
    sample_means = []
    for repetition in range(2000):
        sample_means.append(dmft[rng.choice(N, n_children, replace=False)].mean())
    print("Amostras de", n_children, ": assimetria das médias", round(stats.skew(sample_means), 2), "; desvio padrão", round(np.std(sample_means), 3), "; sigma/sqrt(n):", round(sigma / sqrt(n_children), 3))

Amostras de 5 : assimetria das médias 0.84 ; desvio padrão 1.065 ; sigma/sqrt(n): 1.068
Amostras de 30 : assimetria das médias 0.41 ; desvio padrão 0.452 ; sigma/sqrt(n): 0.436
Amostras de 600 : assimetria das médias 0.05 ; desvio padrão 0.093 ; sigma/sqrt(n): 0.098


### 13.6 O dmft médio e um intervalo de confiança

*Qual é o dmft médio no município, e quão seguros estamos?*

In [6]:
# 1. A amostra de 600: média, desvio padrão e erro padrão
x = srs["dmft"].values
n = len(x)
xbar = x.mean()
s = x.std(ddof=1)
se = s / sqrt(n)
print("Amostra: média", round(xbar, 3), "; desvio padrão", round(s, 3), "; erro padrão", round(se, 4))

# 2. Intervalo de confiança de 95% para a média com o múltiplo t
t_multiple = stats.t.ppf(0.975, n - 1)
low = xbar - t_multiple * se
high = xbar + t_multiple * se
print("Múltiplo t:", round(t_multiple, 3), "; IC de 95%:", round(low, 3), "a", round(high, 3), "; contém a média populacional:", low <= mu <= high)

# 3. Cobertura do intervalo em 2.000 amostras de 600 crianças
rng = np.random.default_rng(13)
hits = 0
for repetition in range(2000):
    sample = dmft[rng.choice(N, 600, replace=False)]
    margin = t_multiple * sample.std(ddof=1) / sqrt(600)
    if abs(sample.mean() - mu) <= margin:
        hits = hits + 1
print("Cobertura com 600 crianças:", round(100 * hits / 2000, 1), "%")

# 4. Amostras de 15 crianças: intervalos com 1,96 e com o múltiplo t, 5.000 amostras
rng = np.random.default_rng(13)
hits_z = 0
hits_t = 0
for repetition in range(5000):
    sample = dmft[rng.choice(N, 15, replace=False)]
    se15 = sample.std(ddof=1) / sqrt(15)
    if abs(sample.mean() - mu) <= 1.96 * se15:
        hits_z = hits_z + 1
    if abs(sample.mean() - mu) <= stats.t.ppf(0.975, 14) * se15:
        hits_t = hits_t + 1
print("Cobertura com 15 crianças: 1,96", round(100 * hits_z / 5000, 1), "%; múltiplo t", round(100 * hits_t / 5000, 1), "%")

Amostra: média 1.847 ; desvio padrão 2.373 ; erro padrão 0.0969
Múltiplo t: 1.964 ; IC de 95%: 1.656 a 2.037 ; contém a média populacional: True
Cobertura com 600 crianças: 95.5 %


Cobertura com 15 crianças: 1,96

 88.4 %; múltiplo t 90.0 %


### 13.7 Parcelas de crianças e os intervalos de Wald e de Wilson

*Que parcela das crianças está livre de cárie, ou tem cárie grave, e quão seguros estamos?*

In [7]:
# 1. Parcelas na amostra de 600: livres de cárie e dmft de 8 ou mais
z = 1.959964
for label, count in (("Livres de cárie:", int((x == 0).sum())), ("Graves (dmft 8 ou mais):", int((x >= 8).sum()))):
    share = count / n
    se_share = sqrt(share * (1 - share) / n)

    # 2. Intervalos de Wald e de Wilson de 95%
    wald_low = share - z * se_share
    wald_high = share + z * se_share
    centre = (share + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * sqrt(share * (1 - share) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    print(label, count, "de", n, "; parcela", round(100 * share, 1), "%; erro padrão", round(se_share, 4))
    print("Wald:", round(100 * wald_low, 1), "a", round(100 * wald_high, 1), "; Wilson:", round(100 * (centre - half), 1), "a", round(100 * (centre + half), 1))

# 3. Cobertura exata dos dois intervalos para amostras de 100 crianças sorteadas sem reposição (hipergeométrica) quando a parcela verdadeira é a de cárie grave
p_true = (dmft >= 8).mean()
counts = np.arange(0, 101)
shares = counts / 100
wald_l = shares - z * np.sqrt(shares * (1 - shares) / 100)
wald_h = shares + z * np.sqrt(shares * (1 - shares) / 100)
wilson_c = (shares + z ** 2 / 200) / (1 + z ** 2 / 100)
wilson_h = z * np.sqrt(shares * (1 - shares) / 100 + z ** 2 / 40000) / (1 + z ** 2 / 100)
probabilities = stats.hypergeom.pmf(counts, N, int((dmft >= 8).sum()), 100)
print("Parcela verdadeira:", round(100 * p_true, 2), "%; cobertura Wald:", round(100 * probabilities[(wald_l <= p_true) & (p_true <= wald_h)].sum(), 1), "%; Wilson:", round(100 * probabilities[(wilson_c - wilson_h <= p_true) & (p_true <= wilson_c + wilson_h)].sum(), 1), "%")

Livres de cárie: 229 de 600 ; parcela 38.2 %; erro padrão 0.0198
Wald: 34.3 a 42.1 ; Wilson: 34.4 a 42.1
Graves (dmft 8 ou mais): 27 de 600 ; parcela 4.5 %; erro padrão 0.0085
Wald: 2.8 a 6.2 ; Wilson: 3.1 a 6.5
Parcela verdadeira: 3.58 %; cobertura Wald: 87.5 %; Wilson: 97.3 %


### 13.8 Estratificando pelo tipo de escola

*As escolas particulares e públicas devem ser amostradas separadamente?*

In [8]:
# 1. Os dois estratos: tamanhos, pesos, médias e desvios padrão
public = df[df["school_type"] == "public"]["dmft"].values
private = df[df["school_type"] == "private"]["dmft"].values
sizes = np.array([len(public), len(private)])
weights = sizes / N
sds = np.array([public.std(ddof=0), private.std(ddof=0)])
print("Públicas:", sizes[0], "crianças; média", round(public.mean(), 3), "; desvio padrão", round(sds[0], 3))
print("Particulares:", sizes[1], "crianças; média", round(private.mean(), 3), "; desvio padrão", round(sds[1], 3))

# 2. Erro padrão da média do município e da média de cada estrato para três alocações
optimal = np.round(600 * weights * sds / (weights * sds).sum())
for name, allocation in (("Proporcional", np.round(600 * weights)), ("Ótima", optimal), ("Igual", np.array([300, 300]))):
    variance = (weights ** 2 * sds ** 2 / allocation * (1 - allocation / sizes)).sum()
    se_h = sds / np.sqrt(allocation) * np.sqrt(1 - allocation / sizes)
    print(name, "alocação", allocation, "; EP total", round(sqrt(variance), 4), "; EP públicas", round(se_h[0], 3), "; EP particulares", round(se_h[1], 3))

# 3. A alocação proporcional pela fórmula: 600 vezes o peso de cada estrato
exact = 600 * weights
print("Alocação proporcional pela fórmula: públicas", round(exact[0], 1), "e particulares", round(exact[1], 1))

Públicas: 10851 crianças; média 2.148 ; desvio padrão 2.562
Particulares: 3448 crianças; média 0.997 ; desvio padrão 1.415
Proporcional alocação [455. 145.] ; EP total 0.0934 ; EP públicas 0.118 ; EP particulares 0.115
Ótima alocação [510.  90.] ; EP total 0.0912 ; EP públicas 0.111 ; EP particulares 0.147
Igual alocação [300 300] ; EP total 0.1123 ; EP públicas 0.146 ; EP particulares 0.078
Alocação proporcional pela fórmula: públicas 455.3 e particulares 144.7


### 13.9 Sorteando escolas em vez de crianças

*O levantamento pode visitar só algumas das escolas?*

In [9]:
# 1. 10.000 amostras de 20 escolas (mesma probabilidade) com 30 crianças cada
schools = np.array(sorted(df["school"].unique()))
rows_of = {}
for school in schools:
    rows_of[school] = np.where(df["school"].values == school)[0]
rng = np.random.default_rng(13)
school_means = []
for repetition in range(10000):
    chosen = rng.choice(schools, 20, replace=False)
    children = np.concatenate([rng.choice(rows_of[school], 30, replace=False) for school in chosen])
    school_means.append(dmft[children].mean())
school_means = np.array(school_means)
deff = (school_means.std() / srs_se) ** 2
print("Escolas: desvio padrão das médias", round(school_means.std(), 4), "; média das médias", round(school_means.mean(), 3), "; efeito de delineamento", round(deff, 2), "; tamanho efetivo da amostra", round(600 / deff))

# 2. Correlação intraconglomerado (análise de variância) e a regra prática para 30 crianças por escola
by_school = df.groupby("school")["dmft"].agg(["mean", "size"])
between = ((by_school["mean"] - mu) ** 2 * by_school["size"]).sum() / (len(schools) - 1)
within = ((df["dmft"] - df.groupby("school")["dmft"].transform("mean")) ** 2).sum() / (N - len(schools))
m0 = (N - (by_school["size"] ** 2).sum() / N) / (len(schools) - 1)
rho = (between - within) / (between + (m0 - 1) * within)
print("Correlação intraconglomerado:", round(rho, 3), "; regra prática 1 + (30 - 1) rho:", round(1 + 29 * rho, 2))

# 3. Escolas sorteadas com probabilidade proporcional ao tamanho (10.000 amostras)
school_sizes = by_school["size"].reindex(schools).values
print("Média das 48 médias escolares:", round(by_school["mean"].mean(), 3), "; média populacional das crianças:", round(mu, 3))
rng = np.random.default_rng(13)
pps_means = []
for repetition in range(10000):
    chosen = rng.choice(schools, 20, replace=True, p=school_sizes / school_sizes.sum())
    children = np.concatenate([rng.choice(rows_of[school], 30, replace=False) for school in chosen])
    pps_means.append(dmft[children].mean())
print("Proporcional ao tamanho: média das médias", round(np.mean(pps_means), 3), "; desvio padrão", round(np.std(pps_means), 3), "; efeito de delineamento", round((np.std(pps_means) / srs_se) ** 2, 2))

Escolas: desvio padrão das médias 0.1391 ; média das médias 1.847 ; efeito de delineamento 2.12 ; tamanho efetivo da amostra 283
Correlação intraconglomerado: 0.066 ; regra prática 1 + (30 - 1) rho: 2.91
Média das 48 médias escolares: 1.846 ; média populacional das crianças: 1.87


Proporcional ao tamanho: média das médias 1.871 ; desvio padrão 0.166 ; efeito de delineamento 3.04


### 13.10 Quando não há lista de crianças

*Quando não há lista de crianças, o que se pode fazer?*

In [10]:
# 1. Amostra por cotas: as primeiras 150 crianças particulares e as primeiras 450 públicas da lista
private_rows = np.where(df["school_type"].values == "private")[0]
public_rows = np.where(df["school_type"].values == "public")[0]
quota = np.concatenate([dmft[private_rows[:150]], dmft[public_rows[:450]]])
print("Amostra por cotas: média", round(quota.mean(), 3), "; livres de cárie:", round(100 * (quota == 0).mean(), 1), "%; escolas:", df.iloc[np.concatenate([private_rows[:150], public_rows[:450]])]["school"].nunique())

Amostra por cotas: média 1.718 ; livres de cárie: 38.3 %; escolas: 3


### 13.11 Quantas crianças precisam ser examinadas?

*Quantas crianças precisam ser examinadas?*

In [11]:
# 1. Tamanho da amostra para a média (margem 0,15), sem e com a correção para população finita
z = 1.959964
n0 = (z * sigma / 0.15) ** 2
n_corrected = n0 / (1 + (n0 - 1) / N)
print("Média:", round(n0, 1), "; com a correção", round(n_corrected, 1))

# 2. Tamanho da amostra para a parcela livre de cárie (margem 0,03) e para p = 0,5
p_free = (dmft == 0).mean()
n_share = z ** 2 * p_free * (1 - p_free) / 0.03 ** 2
print("Parcela:", round(n_share, 1), "; p = 0,5:", round(z ** 2 * 0.25 / 0.03 ** 2, 1))

# 3. Delineamentos estratificados (proporcional e ótimo), margem 0,15, sem e com a correção para população finita
n_proportional = (z / 0.15) ** 2 * (weights * sds ** 2).sum()
n_optimal = (z / 0.15) ** 2 * (weights * sds).sum() ** 2
n_proportional_c = n_proportional / (1 + n_proportional / N)
n_optimal_c = (weights * sds).sum() ** 2 / (0.15 ** 2 / z ** 2 + (weights * sds ** 2).sum() / N)
print("Sem a correção: aleatória simples", int(np.ceil(n0)), "; proporcional", int(np.ceil(n_proportional)), "; ótima", int(np.ceil(n_optimal)))
print("Com a correção: aleatória simples", int(np.ceil(n_corrected)), "; proporcional", int(np.ceil(n_proportional_c)), "; ótima", int(np.ceil(n_optimal_c)))

# 4. Margem de erro com 600 crianças: amostra aleatória simples e 20 escolas de 30
print("Margem de erro: amostra aleatória simples", round(z * srs_se, 3), "; escolas", round(z * school_means.std(), 3))

# 5. Visitar as 48 escolas e examinar 20 ou 30 crianças em cada uma (médias das escolas ponderadas pelo tamanho)
school_sd = df.groupby("school")["dmft"].std(ddof=0).values
school_n = df.groupby("school").size().values
for m in (20, 30):
    variance_m = ((school_n / N) ** 2 * school_sd ** 2 / m * (1 - m / school_n)).sum()
    print("Todas as 48 escolas,", m, "crianças em cada:", 48 * m, "crianças; margem", round(z * sqrt(variance_m), 3))

Média: 974.0 ; com a correção 912.0
Parcela: 980.2 ; p = 0,5: 1067.1
Sem a correção: aleatória simples 975 ; proporcional 933 ; ótima 892
Com a correção: aleatória simples 912 ; proporcional 876 ; ótima 838
Margem de erro: amostra aleatória simples 0.187 ; escolas 0.273
Todas as 48 escolas, 20 crianças em cada: 960 crianças; margem 0.145
Todas as 48 escolas, 30 crianças em cada: 1440 crianças; margem 0.117


## Parte B. Soluções dos exercícios

### 13.12 Exercício 1. As primeiras 600 crianças

In [12]:
# 1. As primeiras 600 crianças e a lista inteira
print("Primeiras 600: média", round(dmft[:600].mean(), 3), "; livres de cárie", round(100 * (dmft[:600] == 0).mean(), 1), "%; escolas", list(df["school"].iloc[:600].unique()))
print("Lista inteira: média", round(mu, 3), "; livres de cárie", round(100 * (dmft == 0).mean(), 1), "%")

Primeiras 600: média 1.295 ; livres de cárie 44.0 %; escolas ['S01', 'S02', 'S03']
Lista inteira: média 1.87 ; livres de cárie 35.7 %


### 13.12 Exercício 2. Uma amostra aleatória simples

In [13]:
# 1. A amostra de 600 com semente 13
print("Média:", round(x.mean(), 3), "; desvio padrão:", round(x.std(ddof=1), 3), "; erro padrão:", round(x.std(ddof=1) / sqrt(600), 4))

Média: 1.847 ; desvio padrão: 2.373 ; erro padrão: 0.0969


### 13.12 Exercício 3. Muitas amostras aleatórias simples

In [14]:
# 1. Médias de 2.000 amostras contra as fórmulas do erro padrão
print("Mean of the means:", round(means600.mean(), 3), "; desvio padrão:", round(means600.std(), 4), "; sigma/sqrt(n):", round(sigma / sqrt(600), 4), "; com a correção:", round(srs_se, 4))

Mean of the means: 1.873 ; desvio padrão: 0.093 ; sigma/sqrt(n): 0.0975 ; com a correção: 0.0954


### 13.12 Exercício 4. Dividir por n ou por n - 1

In [15]:
# 1. Variâncias em 5.000 amostras de 10 crianças
print("Variância populacional:", round(sigma ** 2, 3), "; dividindo por n:", round(np.mean(variances_n), 3), "; dividindo por n - 1:", round(np.mean(variances_n1), 3))

Variância populacional: 5.705 ; dividindo por n: 5.133 ; dividindo por n - 1: 5.704


### 13.12 Exercício 5. Assimetria das médias

In [16]:
# 1. Assimetria das médias para cada tamanho de amostra
for n_children in (5, 30, 600):
    rng = np.random.default_rng(13)
    sample_means = []
    for repetition in range(2000):
        sample_means.append(dmft[rng.choice(N, n_children, replace=False)].mean())
    print("Amostras de", n_children, ": skewness", round(stats.skew(sample_means), 2))

Amostras de 5 : skewness 0.84
Amostras de 30 : skewness 0.41
Amostras de 600 : skewness 0.05


### 13.12 Exercício 6. Intervalo de confiança para a média

In [17]:
# 1. Intervalo para a amostra de 600 e cobertura em 2.000 amostras
print("95% CI:", round(low, 3), "a", round(high, 3), "; contains", mu.round(3), ":", low <= mu <= high, "; coverage in 2,000 samples:", round(100 * hits / 2000, 1), "%")

95% CI: 1.656 a 2.037 ; contains 1.87 : True ; coverage in 2,000 samples: 95.5 %


### 13.12 Exercício 7. Intervalos de Wald e de Wilson

In [18]:
# 1. Intervalos de Wald e de Wilson para as duas parcelas
for label, count in (("Parcela livre de cárie:", int((x == 0).sum())), ("Grave:", int((x >= 8).sum()))):
    share = count / 600
    wald = z * sqrt(share * (1 - share) / 600)
    centre = (share + z ** 2 / 1200) / (1 + z ** 2 / 600)
    half = z * sqrt(share * (1 - share) / 600 + z ** 2 / 1440000) / (1 + z ** 2 / 600)
    print(label, round(100 * share, 1), "%; Wald", round(100 * (share - wald), 1), "a", round(100 * (share + wald), 1), "; Wilson", round(100 * (centre - half), 1), "a", round(100 * (centre + half), 1))

Parcela livre de cárie: 38.2 %; Wald 34.3 a 42.1 ; Wilson 34.4 a 42.1
Grave: 4.5 %; Wald 2.8 a 6.2 ; Wilson 3.1 a 6.5


### 13.12 Exercício 8. Alocações

In [19]:
# 1. Erros padrão para as três alocações
for name, allocation in (("Proporcional", np.round(600 * weights)), ("Ótima", optimal), ("Igual", np.array([300, 300]))):
    variance = (weights ** 2 * sds ** 2 / allocation * (1 - allocation / sizes)).sum()
    se_h = sds / np.sqrt(allocation) * np.sqrt(1 - allocation / sizes)
    print(name, "alocação", allocation, "; EP total", round(sqrt(variance), 4), "; EP públicas", round(se_h[0], 3), "; EP particulares", round(se_h[1], 3))

Proporcional alocação [455. 145.] ; EP total 0.0934 ; EP públicas 0.118 ; EP particulares 0.115
Ótima alocação [510.  90.] ; EP total 0.0912 ; EP públicas 0.111 ; EP particulares 0.147
Igual alocação [300 300] ; EP total 0.1123 ; EP públicas 0.146 ; EP particulares 0.078


### 13.12 Exercício 9. Sorteando escolas

In [20]:
# 1. Efeito de delineamento e tamanho efetivo de 20 escolas de 30 crianças
print("Standard deviation of the means:", round(school_means.std(), 4), "; efeito de delineamento:", round(deff, 2), "; tamanho efetivo da amostra:", round(600 / deff))

Standard deviation of the means: 0.1391 ; efeito de delineamento: 2.12 ; tamanho efetivo da amostra: 283


### 13.12 Exercício 10. Uma amostra por cotas

In [21]:
# 1. A amostra por cotas
print("Média:", round(quota.mean(), 3), "; livres de cárie:", round(100 * (quota == 0).mean(), 1), "%")

Média: 1.718 ; livres de cárie: 38.3 %


### 13.12 Exercício 11. Tamanhos de amostra

In [22]:
# 1. Tamanhos de amostra para a média (margem 0,20) e a parcela (margem 0,04)
n_mean = (z * sigma / 0.20) ** 2
n_share2 = z ** 2 * p_free * (1 - p_free) / 0.04 ** 2
print("Média:", round(n_mean, 1), "; com a correção", round(n_mean / (1 + (n_mean - 1) / N), 1), "; parcela:", round(n_share2, 1), "; com a correção", round(n_share2 / (1 + (n_share2 - 1) / N), 1))

Média: 547.9 ; com a correção 527.7 ; parcela: 551.3 ; com a correção 530.9


### 13.12 Exercício 12. Dois delineamentos com 800 crianças

In [23]:
# 1. Margens de erro de dois delineamentos com 800 crianças
se800 = sigma / sqrt(800) * sqrt((N - 800) / (N - 1))
print("Margem: 800 ao acaso", round(z * se800, 3), "; 20 escolas de 40 (efeito de delineamento 2,3)", round(z * se800 * sqrt(2.3), 3))

Margem: 800 ao acaso 0.161 ; 20 escolas de 40 (efeito de delineamento 2,3) 0.244


---
*Dados sintéticos (semente 13, `generate_data.py`). Valores apenas didáticos; nada aqui é estimativa para um município real.*